In [9]:
from datasets import load_dataset

# MiniStoryGPT

## Introduction
This notebook presents MiniStoryGPT, a custom built language model created for educational purposes,to gain practical experience in training GPT architectures from scratch.

This project aims to implement the principles of the Transformer architecture, as introduced in "Attention is All You Need". Inspired by Andrej Karparthy;s "Zero to Hero" course and his Building GPT from scratch. 

This project leverages the TinyStories dataset to create a model capable of generating coherent, child-friendly narratived.



References:
- Karpathy’s Zero to Hero Course: YouTube
- nanoGPT GitHub Repo: https://github.com/karpathy/nanoGPT
- TinyStories Dataset: https://huggingface.co/datasets/roneneldan/TinyStories
- Attention is All You Need: https://arxiv.org/abs/1706.03762
- TinyStories Paper: https://arxiv.org/abs/2305.07759

## Results

The model, with approximately 20.73 million parameters, was trained on a subset of 1 million TinyStories examples, achieving a training loss of approximately 1.55 after 50,000 iterations

In [10]:
dataset = load_dataset("roneneldan/TinyStories")
len(dataset['train'])

2119719

In [11]:
import torch
import torch.nn as nn
import torch.nn.functional as F


In [12]:
B, T, C = 1, 3, 4 # batch size, sequence length, embed dim
head_size = 4

x = torch.randn(B, T, C)
print("Input x:", x, x.shape)


Input x: tensor([[[ 2.6434,  1.0025, -0.9882,  0.1962],
         [-1.7987, -0.5822,  1.4087, -0.5391],
         [-0.2401, -0.3732, -1.2670, -1.1208]]]) torch.Size([1, 3, 4])


In [13]:
#Single head self-attention
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)

#Compute our projections
K = key(x)
Q = query(x)
V = value(x)

print("\nK:", K)
print("Q:", Q)
print("V:", V)


K: tensor([[[-0.0794, -0.3897,  1.0512,  1.1571],
         [-0.1357,  0.2495, -0.6293, -1.0140],
         [-0.1309,  0.8948,  0.0833,  0.8110]]], grad_fn=<UnsafeViewBackward0>)
Q: tensor([[[ 0.6224,  0.6620,  0.3015,  0.8593],
         [ 0.0864, -0.5104, -0.4490, -0.7908],
         [-0.3395, -0.4087,  0.3145, -0.3703]]], grad_fn=<UnsafeViewBackward0>)
V: tensor([[[ 0.6769, -0.6480, -0.6804,  0.7600],
         [-0.1323,  0.4882,  0.8109, -0.6585],
         [-0.5819,  0.3889, -0.3873,  0.5579]]], grad_fn=<UnsafeViewBackward0>)


In [14]:
#Scale dot-product attention
wei = Q @ K.transpose(-2, -1) * head_size**-0.5 # (B, T, head_size) @ (B, head_size, T) -> (B, T, T)
print("\nRaw attention scores:", wei)

# Softmax → probabilities
attn = F.softmax(wei, dim=-1)
print("\nAttention weights (softmax):", attn)

# Weighted sum of values
out = attn @ V
print("\nOutput of attention head:", out, out.shape)



Raw attention scores: tensor([[[ 0.5019, -0.4902,  0.6165],
         [-0.5975,  0.4727, -0.5734],
         [ 0.0442,  0.0608, -0.2977]]], grad_fn=<MulBackward0>)

Attention weights (softmax): tensor([[[0.4013, 0.1488, 0.4500],
         [0.2024, 0.5902, 0.2074],
         [0.3667, 0.3728, 0.2605]]], grad_fn=<SoftmaxBackward0>)

Output of attention head: tensor([[[-0.0099, -0.0124, -0.3266,  0.4580],
         [-0.0617,  0.2376,  0.2606, -0.1191],
         [ 0.0473,  0.0457, -0.0480,  0.1785]]], grad_fn=<UnsafeViewBackward0>) torch.Size([1, 3, 4])


In [15]:
import torch
import torch.nn.functional as F

z = torch.tensor([2.0, 1.0, 0.1], requires_grad=True)
s = F.softmax(z, dim=0)

# Suppose we want gradient wrt some loss
loss = s[0]  # pick first output
loss.backward()
print(z.grad)  # gradient of softmax output wrt z


tensor([ 0.2247, -0.1598, -0.0650])


In [16]:
import torch
import tiktoken
from model import GPTLanguageModel, device, remap_tokens, custom_decode

# Load mappings and vocab size
old_to_new = torch.load('data/old_to_new.pt', map_location=device)
new_to_old = torch.load('data/new_to_old.pt', map_location=device)
vocab_size = len(old_to_new)

In [17]:
# Tokenizer
enc = tiktoken.get_encoding("gpt2")

# Load model
model = GPTLanguageModel(vocab_size)
checkpoint = torch.load('model_v1.pth', map_location=device)
model.load_state_dict(checkpoint['model_state_dict'])
model = model.to(device)
model.eval()


prompt = "Once upon a time,"

print(f"Generating story starting with: '{prompt}'")

original_context = enc.encode(prompt)
remapped_context = remap_tokens(original_context, old_to_new)
context = torch.tensor([remapped_context], dtype=torch.long, device=device)

with torch.no_grad():
    output = model.generate(context, max_new_tokens=500)
    story = custom_decode(output[0].tolist(), new_to_old, enc)
    print(story)

Generating story starting with: 'Once upon a time,'
Once upon a time, there was a little boy named Timmy. Timmy loved to go to the park with his mommy. One day, while they were playing on a slide, Timmy had an idea. He wanted to swing down the slide all by himself, but his mommy said, "No, Timmy. That's too high, okay?"

They got on the slide and Timmy waited for his turn. He pushed his feet and went down the slide very fast. When he was done, his mommy said, "Now we can go down the slide again!" Timmy was so happy and excited. But then, they heard a loud yell coming from the other side of the slide. They looked down and saw Timmy was scared and asked him what was wrong. Timmy told his mommy about the sharp slide and how he scared his mommy. His mommy encouraged him to be brave and find out the sharp slide. They waited and waited, and eventually after resting, Timmy started to go down the slide. It was uncomfortable at first, but after a few tries, he was very happy that he wasn't scar

## Sample Story
Once upon a time, there was a little girl named Lily. She loved to dance and play with her friends. One day, while playing in the park, she saw a big slide. She wanted to go down it, so she sat up on the slide. But a big dog was barking loudly at her. Lily was scared and didn't know what to do.

Suddenly, a loyal dog named Max came to the park and saw Lily sitting on the slide. He knew something was wrong and wanted to help. He jumped onto the slide and slid down. From then on, Lily always made sure to check the ground before she went down the slide.

The loyal dog was very grateful to Timmy for being so loyal. He also learned that his friends were there to support him. From that day on, Lily and Max played together happily ever after.Once upon a time, there was a big castle. In the castle lived a queen and her friends, the queen and the queen. They all liked to play together and have fun.

One day, the queen was talking to her friends. She said, "Let's play a game with the king and queen!" The queen thought about it and decided to play a game for the king. She said, "Okay, let's play chess together."

The queen and the queen took turns playing chess. They had so much fun. They laughed and even made a game with their friends. The queen and her friends learned about the kingdom in the castle. The queen and her friends continued to play chess together every day.Once upon a time, there was a little girl named Lily. She loved to play in the park with her friends. One day, her mommy said it was time to leave for a lecture about being naughty. Lily didn't want to go, she just wanted to stay home. 

Her mommy said, "Lily, please come back before we leave." But Lily didn't listen and went back to playing. Later that day, Lily's mommy reminded her that she was being naughty and disobedient. She said, "But please behave and play with your toys when you come home so we can go now." 

Lily thought about it and realized her mommy was right. She apologized to her mommy and promised to never behave again. From that day on, she learned to behave and play nicely with her friends.

In [ ]:
#Comparing with Qwen3-0.6B-Base

#!pip install transformers

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load the tokenizer and model
model_name = "Qwen/Qwen3-0.6B-Base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

# Set the prompt
prompt = "Once upon a time there was a boy "

# Tokenize the prompt
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

# Generate the story with 300 new tokens
output = model.generate(
    **inputs,
    max_new_tokens=300,
    do_sample=True,
    temperature=0.7,
    top_p=0.95,
    repetition_penalty=1.2,
    no_repeat_ngram_size=3,
)


generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
print(generated_text)

Once upon a time there was a boy 10 years old. He was very naughty. One day he was playing with his brother. He saw his brother eat a fruit. He was very angry. He ran up to his brother and said, "What are you eating? You are eating a fruit." The boy's brother was very angry and he ran away quickly. The boy ran after him. He came back and said to him, "You are eating a fruit. Do you know what a fruit is?" The boy's brother was very angry and said, "No, I don't know. You are eating a fruit." He ran away quickly. The boy ran after him. He came back and said to him, "You are eating a fruit. Do you know what a fruit is?" He ran away quickly. The boy ran after him. He came back and said to him, "You are eating a fruit. Do you know what a fruit is?" He ran away quickly. The boy ran after him. He came back and said to him, "You are eating a fruit. Do you know what a fruit is?" He ran away quickly. The boy ran after him. He came back and said to him, "You are eating a fruit. Do you know what a fruit is?" He ran away quickly. The boy ran after him. He came back and said to him, "You are eating a fruit. Do you know what a fruit is?" He ran away quickly. The boy ran after him.